In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFacePipeline
from langchain_classic.vectorstores import FAISSb
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document

c:\lang\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\lang\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
video_id = "kCc8FmEb1nY" #Andrej Karpathy's gpt buliding video
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

hi everyone so by now you have probably heard of chat GPT it has taken the world and AI Community by storm and it is a system that allows you to interact with an AI and give it text based tasks so for example we can ask chat GPT to write us a small Hau about how important it is that people understand Ai and then they can use it to improve the world and make it more prosperous so when we run this AI knowledge brings prosperity for all to see Embrace its power okay not bad and so you could see that chpt went from left to right and generated all these words SE sort of sequentially now I asked it already the exact same prompt a little bit earlier and it generated a slightly different outcome ai's power to grow ignorance holds us back learn Prosperity weights so uh pretty good in both cases and slightly different so you can see that chat GPT is a probabilistic system and for any one prompt it can give us multiple answers sort of uh replying to it now this is just one example of a problem pe

In [3]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='hi everyone so by now you have probably', start=0.199, duration=4.721), FetchedTranscriptSnippet(text='heard of chat GPT it has taken the world', start=2.52, duration=5.04), FetchedTranscriptSnippet(text='and AI Community by storm and it is a', start=4.92, duration=4.92), FetchedTranscriptSnippet(text='system that allows you to interact with', start=7.56, duration=5.28), FetchedTranscriptSnippet(text='an AI and give it text based tasks so', start=9.84, duration=5.24), FetchedTranscriptSnippet(text='for example we can ask chat GPT to write', start=12.84, duration=4.12), FetchedTranscriptSnippet(text='us a small Hau about how important it is', start=15.08, duration=3.68), FetchedTranscriptSnippet(text='that people understand Ai and then they', start=16.96, duration=3.2), FetchedTranscriptSnippet(text='can use it to improve the world and make', start=18.76, duration=4.96), FetchedTranscriptSnippet(text='it more prosperous so when 

In [4]:
print(type(transcript))
print(type(transcript_list))

<class 'str'>
<class 'youtube_transcript_api._transcripts.FetchedTranscript'>


In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_text(transcript)

In [6]:
chunks 

["hi everyone so by now you have probably heard of chat GPT it has taken the world and AI Community by storm and it is a system that allows you to interact with an AI and give it text based tasks so for example we can ask chat GPT to write us a small Hau about how important it is that people understand Ai and then they can use it to improve the world and make it more prosperous so when we run this AI knowledge brings prosperity for all to see Embrace its power okay not bad and so you could see that chpt went from left to right and generated all these words SE sort of sequentially now I asked it already the exact same prompt a little bit earlier and it generated a slightly different outcome ai's power to grow ignorance holds us back learn Prosperity weights so uh pretty good in both cases and slightly different so you can see that chat GPT is a probabilistic system and for any one prompt it can give us multiple answers sort of uh replying to it now this is just one example of a problem"

In [7]:
len(chunks)

136

In [8]:
chunks[100]

"think on that data individually and so that's what feed forward is doing and that's why I've added it here now when I train this the validation LW actually continues to go down now to 2. 24 which is down from 2.28 uh the output still look kind of terrible but at least we've improved the situation and so as a preview we're going to now start to intersperse the communication with the computation and that's also what the Transformer does when it has blocks that communicate and then compute and it groups them and replicates them okay so let me show you what we'd like to do we'd like to do something like this we have a block and this block is is basically this part here except for the cross attention now the block basically intersperses communication and then computation the computation the communication is done using multi-headed selfelf attention and then the computation is done using a feed forward Network on all the tokens independently now what I've added here also is you'll notice"

In [9]:
chunks[:3]

["hi everyone so by now you have probably heard of chat GPT it has taken the world and AI Community by storm and it is a system that allows you to interact with an AI and give it text based tasks so for example we can ask chat GPT to write us a small Hau about how important it is that people understand Ai and then they can use it to improve the world and make it more prosperous so when we run this AI knowledge brings prosperity for all to see Embrace its power okay not bad and so you could see that chpt went from left to right and generated all these words SE sort of sequentially now I asked it already the exact same prompt a little bit earlier and it generated a slightly different outcome ai's power to grow ignorance holds us back learn Prosperity weights so uh pretty good in both cases and slightly different so you can see that chat GPT is a probabilistic system and for any one prompt it can give us multiple answers sort of uh replying to it now this is just one example of a problem"

In [10]:
DOCUMENT_1 = Document(page_content=chunks[0], metadata={})

In [11]:
DOCUMENT_1

Document(metadata={}, page_content="hi everyone so by now you have probably heard of chat GPT it has taken the world and AI Community by storm and it is a system that allows you to interact with an AI and give it text based tasks so for example we can ask chat GPT to write us a small Hau about how important it is that people understand Ai and then they can use it to improve the world and make it more prosperous so when we run this AI knowledge brings prosperity for all to see Embrace its power okay not bad and so you could see that chpt went from left to right and generated all these words SE sort of sequentially now I asked it already the exact same prompt a little bit earlier and it generated a slightly different outcome ai's power to grow ignorance holds us back learn Prosperity weights so uh pretty good in both cases and slightly different so you can see that chat GPT is a probabilistic system and for any one prompt it can give us multiple answers sort of uh replying to it now this

In [12]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(
    documents=[Document(page_content=chunk, metadata={}) for chunk in chunks],
    embedding=embedding_model
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2222.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
vector_store.index_to_docstore_id

{0: 'e2bfecd9-6857-4e53-a47e-55980a539523',
 1: 'ce5d28dc-8866-4c58-88c7-2ecebfa181fe',
 2: 'ee0bf061-2411-41c6-91db-240a282d6f1f',
 3: 'a61f785a-298a-43a6-bed6-05549c8f7d4f',
 4: '8659fc3c-e521-4081-ab89-d669143e1d62',
 5: '4eb461c9-2a17-4018-8e6e-6beab3bc4176',
 6: '1435fd02-5099-4cfc-a250-882874949082',
 7: 'b0fa0aad-b931-487d-b049-8ced37972ba3',
 8: '1d456096-43ca-4c2f-a752-f847ea98cec1',
 9: '2c035796-b219-4cf5-9a50-246e881d6bb0',
 10: 'ce2f0fd6-0dc8-40fe-ba46-4f925d1c37ef',
 11: '2d67062e-3611-43f3-83a7-488c62eec745',
 12: 'f6efe7be-d831-4f69-8cad-c273103be508',
 13: 'eabcac6e-3e15-49f1-b70d-c9a5b19838e1',
 14: '8099538e-3bed-4e7e-a914-19fc5411826a',
 15: '1eb8b356-5c69-4783-88ad-cb1e701f68cb',
 16: '82b0dac4-e953-4106-9479-e192b53944f5',
 17: '0b62e680-69d5-4399-985c-2a309cde1694',
 18: 'a1c4805c-d21e-40dd-a00a-bf8cba984d91',
 19: '8b860332-82a2-4a45-aeb5-f74107f2ed62',
 20: 'df275566-4b6f-4e2f-8177-671b76e8a033',
 21: '454d8335-d34f-41ab-a6c2-77012fe45322',
 22: 'f6e7e57a-7330-

In [14]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "lambda_mult": 0.5})

In [15]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000270BA555940>, search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.5})

In [16]:
retriever.invoke("What is gpt?")

[Document(id='b0fa0aad-b931-487d-b049-8ced37972ba3', metadata={}, page_content="um and it is in a GitHub repository that you can find and it's called nanog GPT so nanog GPT is a repository that you can find in my GitHub and it's a repository for training Transformers um on any given text and what I think is interesting about it because there's many ways to train Transformers but this is a very simple implementation so it's just two files of 300 lines of code each one file defines the GPT model the Transformer and one file trains it on some given Text data set and here I'm showing that if you train it on a open web Text data set which is a fairly large data set of web pages then I reproduce the the performance of gpt2 so gpt2 is an early version of open AI GPT uh from 2017 if I recall correctly and I've only so far reproduced the the smallest 124 million parameter model uh but basically this is just proving that the codebase is correctly arranged and I'm able to load the uh neural netwo

In [17]:
llm = HuggingFacePipeline.from_model_id(model_id='google/gemma-3-1b-it', 
    task='text-generation',
    pipeline_kwargs={'max_new_tokens': 1000, 'temperature': 0.7, 'top_p': 0.95}
)
model = ChatHuggingFace(llm=llm)    


Loading weights: 100%|██████████| 340/340 [00:00<00:00, 1613.66it/s]
Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [18]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context and explain the transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [19]:
question = "is there a mention of the transformer architecture in the video?"
retrieved_docs = retriever.invoke(question)

In [20]:
retrieved_docs

[Document(id='8b860332-82a2-4a45-aeb5-f74107f2ed62', metadata={}, page_content="the these nine characters out of the training set this actually has multiple examples packed into it and uh that's because all of these characters follow each other and so what this thing is going to say when we plug it into a Transformer is we're going to actually simultaneously train it to make prediction at every one of these positions now in the in a chunk of nine characters there's actually eight indiv ual examples packed in there so there's the example that when 18 when in the context of 18 47 likely comes next in a context of 18 and 47 56 comes next in a context of 18 47 56 57 can come next and so on so that's the eight individual examples let me actually spell it out with code so here's a chunk of code to illustrate X are the inputs to the Transformer it will just be the first block size characters y will be the uh next block size characters so it's offset by one and that's because y are the targets

In [21]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"the these nine characters out of the training set this actually has multiple examples packed into it and uh that's because all of these characters follow each other and so what this thing is going to say when we plug it into a Transformer is we're going to actually simultaneously train it to make prediction at every one of these positions now in the in a chunk of nine characters there's actually eight indiv ual examples packed in there so there's the example that when 18 when in the context of 18 47 likely comes next in a context of 18 and 47 56 comes next in a context of 18 47 56 57 can come next and so on so that's the eight individual examples let me actually spell it out with code so here's a chunk of code to illustrate X are the inputs to the Transformer it will just be the first block size characters y will be the uh next block size characters so it's offset by one and that's because y are the targets for each position in the input and then here I'm iterating over all the block\

In [22]:
final_prompt = prompt.invoke({'context': context_text, 'question': question})

In [23]:
final_prompt 

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context and explain the transcript context.\n      If the context is insufficient, just say you don't know.\n\n      the these nine characters out of the training set this actually has multiple examples packed into it and uh that's because all of these characters follow each other and so what this thing is going to say when we plug it into a Transformer is we're going to actually simultaneously train it to make prediction at every one of these positions now in the in a chunk of nine characters there's actually eight indiv ual examples packed in there so there's the example that when 18 when in the context of 18 47 likely comes next in a context of 18 and 47 56 comes next in a context of 18 47 56 57 can come next and so on so that's the eight individual examples let me actually spell it out with code so here's a chunk of code to illustrate X are the inputs to the Transformer it w

In [24]:
answer = model.invoke(final_prompt)
print(answer.content)

Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<bos><start_of_turn>user
You are a helpful assistant.
      Answer ONLY from the provided transcript context and explain the transcript context.
      If the context is insufficient, just say you don't know.

      the these nine characters out of the training set this actually has multiple examples packed into it and uh that's because all of these characters follow each other and so what this thing is going to say when we plug it into a Transformer is we're going to actually simultaneously train it to make prediction at every one of these positions now in the in a chunk of nine characters there's actually eight indiv ual examples packed in there so there's the example that when 18 when in the context of 18 47 likely comes next in a context of 18 and 47 56 comes next in a context of 18 47 56 57 can come next and so on so that's the eight individual examples let me actually spell it out with code so here's a chunk of code to illustrate X are the inputs to the Transformer it will just be

In [25]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [26]:
def format_docs(docs):
    context_text = "\n\n".join(doc.page_content for doc in docs)
    return context_text

In [27]:
parallel_chain = RunnableParallel(
    {
        'context': retriever | RunnableLambda(format_docs),
        'question': RunnablePassthrough()
    }
)

In [28]:
parallel_chain.invoke("What is gpt?")

{'context': "um and it is in a GitHub repository that you can find and it's called nanog GPT so nanog GPT is a repository that you can find in my GitHub and it's a repository for training Transformers um on any given text and what I think is interesting about it because there's many ways to train Transformers but this is a very simple implementation so it's just two files of 300 lines of code each one file defines the GPT model the Transformer and one file trains it on some given Text data set and here I'm showing that if you train it on a open web Text data set which is a fairly large data set of web pages then I reproduce the the performance of gpt2 so gpt2 is an early version of open AI GPT uh from 2017 if I recall correctly and I've only so far reproduced the the smallest 124 million parameter model uh but basically this is just proving that the codebase is correctly arranged and I'm able to load the uh neural network weights that openi has released later so you can take a look at 

In [29]:
parser = StrOutputParser()

In [30]:
main_chain = parallel_chain | prompt | model | parser

In [31]:
main_chain.invoke("Summarize the video")

Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"<bos><start_of_turn>user\nYou are a helpful assistant.\n      Answer ONLY from the provided transcript context and explain the transcript context.\n      If the context is insufficient, just say you don't know.\n\n      what we got now here we have one 1 Z so here 110 do product with these two columns will now give us 2 + 6 which is 8 and 7 + 4 which is 11 and because this is 111 we ended up with the addition of all of them and so basically depending on how many ones and zeros we have here we are basically doing a sum currently of a variable number of these rows and that gets deposited into C So currently we're doing sums because these are ones but we can also do average right and you can start to see how we could do average uh of the rows of B uh sort of in an incremental fashion because we don't have to we can basically normalize these rows so that they sum to one and then we're going to get an average so if we took a and then we did aals aide torch. sum in the um of a in the um one